# Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb

# Step 1: Load dataset

In [2]:
df = pd.read_csv("simulated_sales_data.csv")
df['date'] = pd.to_datetime(df['date'])

#  Step 2: Encode 'sku' and 'store_id'

In [3]:
df['sku'] = df['sku'].astype('category')
df['store_id'] = df['store_id'].astype('category')

In [4]:
# Save mapping for decoding later
sku_mapping = dict(enumerate(df['sku'].cat.categories))
store_mapping = dict(enumerate(df['store_id'].cat.categories))
sku_reverse = {v: k for k, v in sku_mapping.items()}
store_reverse = {v: k for k, v in store_mapping.items()}

In [5]:
# Encode to integers
df['sku'] = df['sku'].cat.codes
df['store_id'] = df['store_id'].cat.codes


In [6]:
# Encode 'weather' and extract time features
df['weather'] = df['weather'].astype('category')
weather_categories = df['weather'].cat.categories
df['weather'] = df['weather'].cat.codes
df['day_of_week'] = df['date'].dt.dayofweek
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month

# Step 3: EDA-based logic for future values

In [7]:
# Most frequent weather per day_of_week
weather_mode_by_day = (
    df.groupby('day_of_week')['weather']
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

In [8]:
# Local event probability per day_of_week
event_prob_by_day = (
    df.groupby('day_of_week')['local_event']
    .mean()
    .to_dict()
)

# Step 4: Prepare training data

In [9]:
features = ['store_id', 'sku', 'weather', 'local_event', 'day_of_week', 'day', 'month']
X = df[features]
y = df['units_sold']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#  Step 5: Train model

In [10]:
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"\n✅ XGBoost Model RMSE: {rmse:.2f}")


✅ XGBoost Model RMSE: 10.02


C:\Users\MANISH CHOUDHARY\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


#  Step 6: Generate future dataset using EDA patterns

In [11]:
future_dates = pd.date_range(start=df['date'].max() + timedelta(days=1), periods=7)
unique_skus = df['sku'].unique()
unique_stores = df['store_id'].unique()

future_data = []

for store_id in unique_stores:
    for sku in unique_skus:
        for date in future_dates:
            dow = date.dayofweek

            # Use most common weather from dataset for this weekday
            weather = weather_mode_by_day.get(dow, 0)

            # Use majority event flag (0 or 1) based on dataset trend
            prob = event_prob_by_day.get(dow, 0.0)
            local_event = 1 if prob >= 0.5 else 0

            future_data.append({
                'store_id': store_id,
                'sku': sku,
                'weather': weather,
                'local_event': local_event,
                'day_of_week': dow,
                'day': date.day,
                'month': date.month
            })

# Create DataFrame
future_df = pd.DataFrame(future_data)
future_df = future_df[features]  # Ensure correct column order

# Step 7: Predict future demand

In [12]:
future_df['predicted_units_sold'] = model.predict(future_df)


In [13]:
# Decode IDs for readability
future_df['store_id'] = future_df['store_id'].map(store_mapping)
future_df['sku'] = future_df['sku'].map(sku_mapping)

In [14]:

# Decode weather label
future_df['weather_label'] = future_df['weather'].map(dict(enumerate(weather_categories)))

# Step 8: Final output

In [15]:
print("\n📦 Forecasted Demand (Next 7 Days):\n")
print(future_df[['store_id', 'sku', 'day', 'month', 'day_of_week', 'weather_label', 'local_event', 'predicted_units_sold']].head(50))


📦 Forecasted Demand (Next 7 Days):

   store_id       sku  day  month  day_of_week weather_label  local_event  \
0      W001   MILK001    3      7            3         Rainy            1   
1      W001   MILK001    4      7            4           Hot            1   
2      W001   MILK001    5      7            5          Cold            0   
3      W001   MILK001    6      7            6           Hot            0   
4      W001   MILK001    7      7            0         Rainy            0   
5      W001   MILK001    8      7            1          Cold            0   
6      W001   MILK001    9      7            2           Hot            0   
7      W001    EGG002    3      7            3         Rainy            1   
8      W001    EGG002    4      7            4           Hot            1   
9      W001    EGG002    5      7            5          Cold            0   
10     W001    EGG002    6      7            6           Hot            0   
11     W001    EGG002    7      7      

# Phase 2 for Waste Alerts & Dynamic Discounting

# Step 1: Simulate stock_on_hand and days_to_expiry

In [16]:
import numpy as np

# Simulate stock_on_hand (between 10 and 30)
np.random.seed(42)
future_df['stock_on_hand'] = np.random.randint(10, 30, size=len(future_df))

# Simulate days_to_expiry (between 1 and 5 days)
future_df['days_to_expiry'] = np.random.randint(1, 5, size=len(future_df))


#  Step 2: Define Waste Risk & Discount Rules

In [17]:
def classify_action(row):
    if row['predicted_units_sold'] <= 10:
        return 'Suggest Donation'
    elif row['stock_on_hand'] > row['predicted_units_sold'] and row['days_to_expiry'] <= 2:
        return 'Trigger Discount'
    elif row['stock_on_hand'] > row['predicted_units_sold']:
        return 'Monitor Overstock'
    else:
        return 'No Action'

# Apply logic to each row
future_df['action'] = future_df.apply(classify_action, axis=1)


In [18]:
print(future_df[['store_id', 'sku', 'predicted_units_sold', 'stock_on_hand', 'days_to_expiry', 'action']])


   store_id      sku  predicted_units_sold  stock_on_hand  days_to_expiry  \
0      W001  MILK001             18.386650             16               3   
1      W001  MILK001             13.681134             29               1   
2      W001  MILK001             15.979356             24               4   
3      W001  MILK001             26.848158             20               2   
4      W001  MILK001             11.306979             17               1   
..      ...      ...                   ...            ...             ...   
65     W002   VEG005             24.092813             23               3   
66     W002   VEG005             20.336014             17               2   
67     W002   VEG005              8.077477             25               3   
68     W002   VEG005             17.237917             22               3   
69     W002   VEG005             18.696068             27               1   

               action  
0           No Action  
1    Trigger Discount  
2  

In [19]:
future_df.head(5)

,store_id,sku,weather,local_event,day_of_week,day,month,predicted_units_sold,weather_label,stock_on_hand,days_to_expiry,action
0,W001,MILK001,2,1,3,3,7,18.386650,Rainy,16,3,No Action
1,W001,MILK001,1,1,4,4,7,13.681134,Hot,29,1,Trigger Discount
2,W001,MILK001,0,0,5,5,7,15.979356,Cold,24,4,Monitor Overstock
3,W001,MILK001,1,0,6,6,7,26.848158,Hot,20,2,No Action
4,W001,MILK001,2,0,0,7,7,11.306979,Rainy,17,1,Trigger Discount


In [26]:
future_df

,store_id,sku,weather,local_event,day_of_week,day,month,predicted_units_sold,weather_label,stock_on_hand,days_to_expiry,action
0,W001,MILK001,2,1,3,3,7,18.386650,Rainy,16,3,No Action
1,W001,MILK001,1,1,4,4,7,13.681134,Hot,29,1,Trigger Discount
2,W001,MILK001,0,0,5,5,7,15.979356,Cold,24,4,Monitor Overstock
3,W001,MILK001,1,0,6,6,7,26.848158,Hot,20,2,No Action
4,W001,MILK001,2,0,0,7,7,11.306979,Rainy,17,1,Trigger Discount
...,...,...,...,...,...,...,...,...,...,...,...,...
65,W002,VEG005,0,0,5,5,7,24.092813,Cold,23,3,No Action
66,W002,VEG005,1,0,6,6,7,20.336014,Hot,17,2,No Action
67,W002,VEG005,2,0,0,7,7,8.077477,Rainy,25,3,Suggest Donation
68,W002,VEG005,0,0,1,8,7,17.237917,Cold,22,3,Monitor Overstock


In [27]:
future_df.to_csv("future_df_final.csv", index=False)

In [28]:
df = pd.read_csv("future_df_final.csv").head(70)
df

,store_id,sku,weather,local_event,day_of_week,day,month,predicted_units_sold,weather_label,stock_on_hand,days_to_expiry,action
0,W001,MILK001,2,1,3,3,7,18.386650,Rainy,16,3,No Action
1,W001,MILK001,1,1,4,4,7,13.681134,Hot,29,1,Trigger Discount
2,W001,MILK001,0,0,5,5,7,15.979356,Cold,24,4,Monitor Overstock
3,W001,MILK001,1,0,6,6,7,26.848158,Hot,20,2,No Action
4,W001,MILK001,2,0,0,7,7,11.306979,Rainy,17,1,Trigger Discount
...,...,...,...,...,...,...,...,...,...,...,...,...
65,W002,VEG005,0,0,5,5,7,24.092813,Cold,23,3,No Action
66,W002,VEG005,1,0,6,6,7,20.336014,Hot,17,2,No Action
67,W002,VEG005,2,0,0,7,7,8.077477,Rainy,25,3,Suggest Donation
68,W002,VEG005,0,0,1,8,7,17.237917,Cold,22,3,Monitor Overstock


# Thank you for the Oppurtunity 